# 12 — Consolidación híbrida de la clasificación (v1 + v2)

Combina las dos iteraciones del clasificador zero-shot en un dataset definitivo.

## Contexto

Se ejecutaron dos iteraciones de BART-MNLI:

- **v1** (`scam_us_FINAL_classified.csv`): etiquetas genéricas. Mayor confianza promedio (0.596) pero solapaba `gov_impersonation` dentro de `phishing_identity` (solo 5 tweets en gov).
- **v2** (`scam_us_FINAL_classified_v2.csv`): etiquetas refinadas. Resolvió `gov_impersonation` (99 tweets) pero con menor confianza global (0.474) y desestabilizó otras categorías (`payment_app` 37→5, `romance` 198→122).

## Regla de consolidación

Para cada tweet:

1. **Por defecto, se usa la categoría de v1** (clasificación base, mayor confianza, categorías estables).
2. **Excepción:** si v1 clasificó como `phishing_identity` Y v2 clasificó como `gov_impersonation` Y la confianza de v2 ≥ 0.40 → se reclasifica como `gov_impersonation`.

Esto corrige el único defecto sistemático de v1 (el solapamiento phishing/gov) sin importar la inestabilidad global de v2.

## Salida

`scam_us_FINAL_classified_hybrid.csv` — dataset definitivo para el análisis del TFM.

**No modifica** los CSV de v1 ni v2 (se conservan para trazabilidad).


## Carga de los dos datasets

In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_colwidth', None)

V1_PATH = "../data/raw/scam_us_FINAL_classified.csv"
V2_PATH = "../data/raw/scam_us_FINAL_classified_v2.csv"

for path in (V1_PATH, V2_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encuentra {path}. Necesario para el híbrido.")

df_v1 = pd.read_csv(V1_PATH)
df_v2 = pd.read_csv(V2_PATH)

print(f"v1 cargado: {len(df_v1)} filas")
print(f"v2 cargado: {len(df_v2)} filas")
print()
print(f"Columnas v1 relevantes: predicted_category, confidence_score")
print(f"Columnas v2 relevantes: predicted_category_v2, confidence_score_v2")


## Cruce de ambos por tweet_id

In [ ]:
# Tomamos v1 como base y le añadimos las columnas de v2
v2_cols = df_v2[["tweet_id", "predicted_category_v2", "confidence_score_v2",
                 "predicted_label_v2"]].copy()

df = df_v1.merge(v2_cols, on="tweet_id", how="left")

print(f"Dataset combinado: {len(df)} filas")

# Comprobar que el merge no perdió filas
n_sin_v2 = df["predicted_category_v2"].isna().sum()
if n_sin_v2 > 0:
    print(f"⚠️ {n_sin_v2} tweets no tienen correspondencia en v2")
else:
    print("✓ Todos los tweets tienen datos de v1 y v2")


## Aplicar la regla de consolidación

In [ ]:
CONFIDENCE_THRESHOLD = 0.40

# Punto de partida: la categoría de v1
df["predicted_category_final"] = df["predicted_category"]
df["confidence_score_final"] = df["confidence_score"]
df["classification_source"] = "v1"

# Condición de excepción: v1=phishing_identity, v2=gov_impersonation, conf_v2>=0.40
mask_reclassify = (
    (df["predicted_category"] == "phishing_identity") &
    (df["predicted_category_v2"] == "gov_impersonation") &
    (df["confidence_score_v2"] >= CONFIDENCE_THRESHOLD)
)

n_reclassified = mask_reclassify.sum()

# Aplicar la reclasificación
df.loc[mask_reclassify, "predicted_category_final"] = "gov_impersonation"
df.loc[mask_reclassify, "confidence_score_final"] = df.loc[mask_reclassify, "confidence_score_v2"]
df.loc[mask_reclassify, "classification_source"] = "v2 (regla gov_impersonation)"

# is_relevant final
df["is_relevant_final"] = df["predicted_category_final"] != "not_related"

print("=== APLICACIÓN DE LA REGLA DE CONSOLIDACIÓN ===\n")
print(f"  Umbral de confianza v2:           {CONFIDENCE_THRESHOLD}")
print(f"  Tweets reclasificados a gov:      {n_reclassified}")
print(f"  (eran phishing_identity en v1)")
print()

# Cuántos casos de frontera había en total y cuántos se descartaron por umbral
mask_frontera = (
    (df["predicted_category"] == "phishing_identity") &
    (df["predicted_category_v2"] == "gov_impersonation")
)
n_frontera_total = mask_frontera.sum()
n_descartados = n_frontera_total - n_reclassified
print(f"  Casos de frontera phishing/gov (total):  {n_frontera_total}")
print(f"  Reclasificados (conf v2 >= {CONFIDENCE_THRESHOLD}):       {n_reclassified}")
print(f"  Descartados (conf v2 < {CONFIDENCE_THRESHOLD}):           {n_descartados}")


## Distribución final del híbrido

In [ ]:
print("=== DISTRIBUCIÓN FINAL (HÍBRIDO) ===\n")
counts = df["predicted_category_final"].value_counts()
total = len(df)
for cat, n_cat in counts.items():
    pct = n_cat / total * 100
    print(f"  {cat:<22} {n_cat:>5}  ({pct:>5.1f}%)")

print(f"\nRelevantes: {df['is_relevant_final'].sum()} / {total} ({df['is_relevant_final'].mean()*100:.1f}%)")


In [ ]:
# Comparativa de las tres versiones lado a lado
print("=== COMPARATIVA v1 / v2 / HÍBRIDO ===\n")
v1_counts = df["predicted_category"].value_counts()
v2_counts = df["predicted_category_v2"].value_counts()
hyb_counts = df["predicted_category_final"].value_counts()

all_cats = sorted(set(v1_counts.index) | set(v2_counts.index) | set(hyb_counts.index))
print(f"  {'categoría':<22} {'v1':>6} {'v2':>6} {'híbrido':>8}")
print("  " + "-" * 46)
for cat in all_cats:
    n1 = v1_counts.get(cat, 0)
    n2 = v2_counts.get(cat, 0)
    nh = hyb_counts.get(cat, 0)
    print(f"  {cat:<22} {n1:>6} {n2:>6} {nh:>8}")


In [ ]:
# Confianza del híbrido
print("=== CONFIANZA DEL HÍBRIDO ===\n")
print(f"  Media:    {df['confidence_score_final'].mean():.3f}")
print(f"  Mediana:  {df['confidence_score_final'].median():.3f}")
print(f"  Min:      {df['confidence_score_final'].min():.3f}")
print(f"  Max:      {df['confidence_score_final'].max():.3f}")
print()
print(f"  Alta confianza (>=0.7):  {(df['confidence_score_final'] >= 0.7).sum()}")
print(f"  Media (0.4-0.7):         {((df['confidence_score_final'] >= 0.4) & (df['confidence_score_final'] < 0.7)).sum()}")
print(f"  Baja (<0.4):             {(df['confidence_score_final'] < 0.4).sum()}")
print()
print("Nota: la confianza del híbrido es esencialmente la de v1,")
print("salvo en los tweets reclasificados, que toman la confianza de v2.")


## Verificación: los tweets reclasificados

In [ ]:
print("=== TWEETS RECLASIFICADOS A gov_impersonation ===\n")
reclassified = df[df["classification_source"] == "v2 (regla gov_impersonation)"]
print(f"Total: {len(reclassified)}\n")
for _, row in reclassified.nlargest(15, "confidence_score_v2").iterrows():
    print(f"  [v2 conf {row['confidence_score_v2']:.2f}] @{row.get('username','')}")
    print(f"    {str(row['text'])[:220]}")
    print()


## Guardado del dataset definitivo

In [ ]:
# Limpiamos columnas: dejamos las finales claras
OUTPUT = "../data/raw/scam_us_FINAL_classified_hybrid.csv"
df.to_csv(OUTPUT, index=False, encoding="utf-8")

print(f"✓ Guardado: {OUTPUT}")
print(f"  Total filas: {len(df)}")
print()
print("Columnas clave del dataset definitivo:")
print("  predicted_category_final   ← categoría definitiva (USAR ESTA)")
print("  confidence_score_final     ← confianza definitiva")
print("  is_relevant_final          ← True si no es not_related")
print("  classification_source      ← 'v1' o 'v2 (regla gov_impersonation)'")
print()
print("Columnas conservadas para trazabilidad:")
print("  predicted_category / confidence_score          (v1 original)")
print("  predicted_category_v2 / confidence_score_v2     (v2 original)")
print("  bertopic_id / bertopic_keywords                 (BERTopic)")
print()
print("📦 Archivos (los tres se conservan):")
print("  scam_us_FINAL_classified.csv          ← v1 (intacto)")
print("  scam_us_FINAL_classified_v2.csv       ← v2 (intacto)")
print("  scam_us_FINAL_classified_hybrid.csv   ← DEFINITIVO")
print()
print("🚨 HAZ COPIA DE SEGURIDAD del hybrid a Drive.")


## Resumen para la memoria del TFM

Texto orientativo para documentar esta decisión metodológica.


In [ ]:
resumen = f"""
RESUMEN METODOLÓGICO (para la memoria)
{'=' * 55}

Se ejecutaron dos iteraciones del clasificador zero-shot BART-MNLI
sobre el corpus de {len(df)} tweets:

- Iteración 1: etiquetas de categoría genéricas. Confianza media
  {df['confidence_score'].mean():.3f}. Limitación: solapamiento sistemático
  entre 'phishing/robo de identidad' y 'suplantación de organismos
  públicos' (esta última infrarrepresentada).

- Iteración 2: etiquetas refinadas con mayor especificidad. Resolvió
  el solapamiento pero con menor confianza global ({df['confidence_score_v2'].mean():.3f})
  y mayor inestabilidad en categorías previamente bien definidas.

Estrategia de consolidación adoptada: la clasificación base es la de
la iteración 1. Se reclasificaron como 'suplantación de organismos
públicos' únicamente los tweets que (a) la iteración 1 había asignado
a 'phishing', (b) la iteración 2 asignaba a 'suplantación de organismos
públicos' y (c) superaban un umbral de confianza de 0.40 en la
iteración 2.

Resultado: {(df['classification_source'] != 'v1').sum()} tweets reclasificados.
La categoría 'suplantación de organismos públicos' pasó de
{(df['predicted_category'] == 'gov_impersonation').sum()} a
{(df['predicted_category_final'] == 'gov_impersonation').sum()} tweets,
manteniéndose la estabilidad del resto de categorías y la confianza
global de la clasificación base.
"""
print(resumen)
